
Load the nanotube + methane example recording as an MDAnalsis universe:

In [ ]:
from nanover.mdanalysis import universe_from_recording
from nanover.trajectory import FrameData

universe = universe_from_recording("gluhut-unbinding.nanover.zip")

Find the centroid of the methane molecule for each frame:

In [ ]:
import numpy as np

METHANE_ATOMS = universe.residues[1].atoms.indices

METHANE_CENTROIDS_ABSOLUTE = []
for i, _ in enumerate(universe.trajectory):
    positions = universe.atoms.positions[METHANE_ATOMS] / 10  # angstrom -> nm
    centroid = np.mean(positions, axis=0)
    METHANE_CENTROIDS_ABSOLUTE.append(centroid)

Approximating the nanotube as a rigid object, find the relative transform between the nanotube's initial position and orientation and it's position and orientation at each frame:

In [ ]:
from nanover.utilities.transforms import Transform, find_transformation_between_point_patterns

NANOTUBE_ATOMS = universe.residues[0].atoms.indices

_ = universe.trajectory[0]
NANOTUBE_INITIAL_POSITIONS = universe.atoms.positions[NANOTUBE_ATOMS] / 10  # angstrom -> nm

NANOTUBE_TRANSFORMS: list[Transform] = []
for _ in universe.trajectory:
    nanotube_positions = universe.atoms.positions[NANOTUBE_ATOMS] / 10  # angstrom -> nm
    matrix = find_transformation_between_point_patterns(NANOTUBE_INITIAL_POSITIONS, nanotube_positions)
    transform = Transform.from_parent_to_local_matrix(matrix)
    NANOTUBE_TRANSFORMS.append(transform)

Set up the server with playback of the universe:

In [ ]:
from nanover.app import OmniRunner
from nanover.mdanalysis import UniverseSimulation

simulation = UniverseSimulation.from_universe(universe, name="nanotube + methane")
simulation.playback_factor = 30
simulation.load()

imd_runner = OmniRunner.with_basic_server(simulation, port=0, name="EXAMPLE:  trajectory seeker (aligned)")
imd_runner.load(0)

Import the jupyter utilities for drawing + interaction:

In [ ]:
from nanover.jupyter import NanoverJupyterUtilities

utilities = NanoverJupyterUtilities.from_runner(imd_runner)

Continuously update a line in the scene that shows the path of the methane centroid relative the current frame's nanotube:

In [ ]:
from nanover.jupyter import FrameListener

class RelativePathVisuals(FrameListener):
    points = []

    def on_frame_update(self, full_frame: FrameData, frame_update: FrameData):
        frame_i_transform = NANOTUBE_TRANSFORMS[simulation.prev_frame]

        self.points = []
        for n, (frame_n_centroid, frame_n_transform) in enumerate(zip(METHANE_CENTROIDS_ABSOLUTE, NANOTUBE_TRANSFORMS)):
            # maintaining relative position to the nanotube, where would this centroid be if the nanotube was in its frame 0 position?
            frame_0_centroid = frame_n_transform.point_local_to_parent(frame_n_centroid)
            # maintaining relative position to the nanotube, where would this centroid be if the nanotube was in its current position?
            frame_i_centroid = frame_i_transform.point_parent_to_local(frame_0_centroid)
            self.points.append(frame_i_centroid)
        utilities.objects.update_line("path", positions=self.points, size=0.01)

visuals = RelativePathVisuals.from_runner(imd_runner)
visuals.start()

In [ ]:
from nanover.jupyter import FrameListener

r1_atoms = np.array([47, 48, 51, 56, 59, 80]) - 1  # top ring
r2_atoms = np.array([3, 4, 5, 8, 25, 28])     - 1  # bottom ring
r3_atoms = np.array([33, 34, 35, 36, 42, 41]) - 1  # exit p3

l1_atom = 163-1
l2_atom = 174-1
l3_atom = 160-1

a1_atoms = [r1_atoms, [l1_atom], [l2_atom]]
a2_atoms = [r2_atoms, r1_atoms, [l1_atom]]

d1_atoms = [r1_atoms, [l1_atom], [l2_atom], [l3_atom]]
d2_atoms = [r2_atoms, r1_atoms, [l1_atom], [l2_atom]]
d3_atoms = [r3_atoms, r2_atoms, r1_atoms, [l1_atom]]

def dihedral_amber(a1, a2, a3, a4):
    """
    Computes the angle between three vectors made up of four points (all three
    vectors share one point with one other vector)

    Parameters
    ----------
    a1, a2, a3, a4 : Atom or collection of 4 coordinates
        The four atoms between whom the torsion angle should be calculated (with
        a1 and a4 being the two end-point atoms not shared between two vectors)

    Returns
    -------
    dihed : float
        The measured dihedral between the 4 points in degrees
    """
    p = np.array([a1,
                  a2,
                  a3,
                  a4])
    v1 = p[1] - p[0]
    v2 = p[1] - p[2]
    v3 = p[3] - p[2]
    # Take the cross product between v1-v2 and v2-v3
    v1xv2 = np.cross(v1, v2)
    v2xv3 = np.cross(v2, v3)
    # Now find the angle between these cross-products
    l1 = np.sqrt(np.dot(v1xv2, v1xv2))
    l2 = np.sqrt(np.dot(v2xv3, v2xv3))
    cosa = np.dot(v1xv2, v2xv3) / (l1 * l2)
    if np.dot(v3, v1xv2) <= 0.0 :
        return np.degrees(np.arccos(cosa))
    else :
        return -np.degrees(np.arccos(cosa))

def angle(a, b, c):
    """ Calculate an angle in degreee given three point a,b,c """
    ba = a - b
    bc = c - b
    cosine_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc))
    angle_degrees = np.degrees(np.arccos(cosine_angle))
    return angle_degrees

def angle_pos(a, b, c):
    delta = (a+c)/2 - b
    dir = delta / np.linalg.norm(delta)
    return b + dir * .1

class CVVisuals(FrameListener):
    def on_frame_update(self, full_frame: FrameData, frame_update: FrameData):
        try:
            r1 = np.mean(full_frame.particle_positions[r1_atoms], axis=0)
            r2 = np.mean(full_frame.particle_positions[r2_atoms], axis=0)
            r3 = np.mean(full_frame.particle_positions[r3_atoms], axis=0)
            l1 = full_frame.particle_positions[l1_atom]
            l2 = full_frame.particle_positions[l2_atom]
            l3 = full_frame.particle_positions[l3_atom]

            r = np.linalg.norm(r1-l1, axis=0)

            with utilities.objects:
                # rings centroids
                for i, position in enumerate([r1, r2, r3]):
                    utilities.objects.update_shape(f"p{i}", position=position, color=[1.0, 1.0, 0.0, 1.0], size=.1)
                # # ligand atoms
                # for i, position in enumerate([l1, l2, l3]):
                #     utilities.objects.update_shape(f"l{i}", position=position, color=[1.0, 1.0, 1.0, 1.0], size=.15)
                # distance
                utilities.objects.update_line("r1l1", positions=[l1, r1], size=0.025)
                utilities.objects.update_label("r1l1", position=(r1+l1)/2, text=f"{r:.2f}nm")
                # angles
                for i, triplet in enumerate([a1_atoms, a2_atoms]):
                    a, b, c = [np.mean(full_frame.particle_positions[atoms], axis=0) for atoms in triplet]
                    utilities.objects.update_line(f"a{i}ab", positions=[a, b], size=0.025, color=[0.0, 1.0, 0.0, .5])
                    utilities.objects.update_line(f"a{i}bc", positions=[b, c], size=0.025, color=[0.0, 1.0, 0.0, .5])

                    # utilities.objects.update_line(f"a{i}m", positions=[abm, bcm], size=0.025, color=[1.0, 0.0, 0.0, .5])
                    utilities.objects.update_label(f"a{i}abc", position=angle_pos(a, b, c), text=f"{angle(a, b, c):.1f}deg")
                # dihedrals
                for i, quad in enumerate([d1_atoms, d2_atoms, d3_atoms]):
                    a, b, c, d = [np.mean(full_frame.particle_positions[atoms], axis=0) for atoms in quad]

                    blue = [0.0, 0.0, 1.0, .5]
                    m = (b+c)/2

                    utilities.objects.update_line(f"a{i}am", positions=[a, m], size=0.025, color=blue)
                    utilities.objects.update_line(f"a{i}dm", positions=[d, m], size=0.025, color=blue)
                    utilities.objects.update_line(f"d{i}bc", positions=[b, c], size=0.025, color=blue)

                    utilities.objects.update_label(f"d{i}", position=angle_pos(a, (b+c)/2, d), text=f"{dihedral_amber(a, b, c, d):.1f}deg")

        except Exception:
            from traceback import format_exc
            utilities.objects.update_label("debug", text=format_exc())

cvvisuals = CVVisuals.from_runner(imd_runner)
cvvisuals.start()

Define a mode that shows the nearest point on the centroid path as the user controllers hover close to it, and seeks the recording when they click:

In [ ]:
from nanover.jupyter import Mode
from intersection import closest_point_on_polyline


class SeekLineMode(Mode):
    def on_button_pressed(self, *, key: str, cursor: dict, button: str):
        if button == "primary":
            next_pos = utilities.scene_transform.point_parent_to_local(cursor["position"])
            result = closest_point_on_polyline(visuals.points, next_pos)
            utilities.notify_all(f"seek to frame #{result.index}")
            simulation.seek_to_frame_index(result.index)

    def on_cursor_updated(self, *, key: str, cursor: dict):
        next_pos = utilities.scene_transform.point_parent_to_local(cursor["position"])
        result = closest_point_on_polyline(visuals.points, next_pos)

        if result.distance < 0.05:
            utilities.objects.update_shape(f"cursor.{key}", position=result.point, size=0.02)
        else:
            utilities.objects.remove_shape(f"cursor.{key}")


utilities.use_interaction_modes()
utilities.add_interaction_mode(SeekLineMode, "seek", icon="⏩")